# Simplex pipeline

Defines a disease set → resolves EFO IDs to UMLS via OpenTargets → loads MSI diffusion profiles → computes barycentric simplex embedding → saves `results/simplex/{name}.*` for use by `simplex_3d.py`.

**Inputs you edit:** the `DISEASE_SET` dict in §1 (labels + EFO IDs) and `NAME`.

**Shortcut:** if you already know the UMLS IDs, skip §2 and fill `profiles_map` directly.

In [ ]:
import sys, os, json, time
import numpy as np
import pandas as pd
import requests

# project root on sys.path so msi / diff_prof imports work from any CWD
ROOT = os.path.abspath("")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from msi.msi import MSI
from diff_prof.diffusion_profiles import DiffusionProfiles

## §1  Define your disease set

Map human-readable labels to EFO / MONDO IDs (OpenTargets format).  
Set `NAME` to choose the output filename under `results/simplex/`.

In [ ]:
NAME = "my_set"   # → results/simplex/my_set.npz / .json

DISEASE_SET = {
    "Breast Carcinoma":     "EFO_0000305",
    "NSCLC":                "EFO_0003060",
    "Alzheimer's":          "EFO_0000249",
    "Type 2 Diabetes":      "EFO_0001360",
    "Rheumatoid Arthritis": "EFO_0000685",
}

OT_URL = "https://api.platform.opentargets.org/api/v4/graphql"

print(f"Disease set '{NAME}':")
for label, efo in DISEASE_SET.items():
    print(f"  {label:<25s}  {efo}")

## §2  Resolve EFO IDs → UMLS IDs via OpenTargets

Queries the OT GraphQL API for each disease node and extracts the UMLS cross-reference.  
**Skip this cell** and fill `profiles_map` manually if you already know the UMLS IDs.

In [ ]:
_QUERY = """
query DiseaseInfo($id: String!) {
  disease(efoId: $id) { id  name  dbXRefs }
}
"""

def ot_disease_info(efo_id):
    r = requests.post(OT_URL, json={"query": _QUERY, "variables": {"id": efo_id.replace(":", "_")}})
    r.raise_for_status()
    return r.json().get("data", {}).get("disease")

def first_umls(db_x_refs):
    for ref in (db_x_refs or []):
        if ref.startswith("UMLS:"):
            return ref.split(":", 1)[1]
    return None


profiles_map = {}   # label → UMLS ID

for label, efo_id in DISEASE_SET.items():
    info = ot_disease_info(efo_id)
    if info is None:
        print(f"  [{label}]  {efo_id} — NOT FOUND in OpenTargets")
        continue
    umls = first_umls(info.get("dbXRefs"))
    if umls is None:
        print(f"  [{label}]  {efo_id} ({info['name']}) — no UMLS xref")
        continue
    profiles_map[label] = umls
    print(f"  [{label}]  {efo_id} → {umls}  ({info['name']})")

print(f"\nResolved {len(profiles_map)} / {len(DISEASE_SET)} labels to UMLS IDs")

## §3  Load MSI and diffusion profiles

In [ ]:
print("Loading MSI...")
t0 = time.time()
msi = MSI()
msi.load()
print(f"  {len(msi.nodelist):,} nodes  ({time.time()-t0:.1f}s)")

print("Loading diffusion profiles...")
t0 = time.time()
dp = DiffusionProfiles(
    alpha=None, max_iter=None, tol=None, weights=None,
    num_cores=None, save_load_file_path="results/",
)
msi.load_saved_node_idx_mapping_and_nodelist(dp.save_load_file_path)
dp.load_diffusion_profiles(msi.drugs_in_graph + msi.indications_in_graph)
print(f"  {len(dp.drug_or_indication2diffusion_profile):,} profiles loaded  ({time.time()-t0:.1f}s)")

In [ ]:
# Validate every UMLS ID has a saved profile
p = dp.drug_or_indication2diffusion_profile

ok, missing = [], []
for label, umls in profiles_map.items():
    (ok if umls in p else missing).append((label, umls))

for label, umls in ok:
    print(f"  ✓  {label:<25s}  {umls}")
for label, umls in missing:
    print(f"  ✗  {label:<25s}  {umls}  — no saved profile")

if missing:
    raise ValueError(
        f"{len(missing)} UMLS ID(s) have no diffusion profile: "
        f"{[u for _, u in missing]}\n"
        "Check the ID is present in msi.indications_in_graph, or run "
        "dp.calculate_diffusion_profiles(msi) first."
    )

print(f"\nAll {len(ok)} profiles found.")

## §4  Compute barycentric simplex embedding

For each node $v$ and K profiles $\{P_k\}$:

$$W_k(v) = \frac{P_k(v)}{\sum_k P_k(v)}, \qquad
  \tilde{W}_k(v) = W_k(v) - 1/K$$

Relevance (KL contribution vs uniform): $M(v) \cdot \log(N \cdot M(v))$, where $M(v) = \text{mean}_k P_k(v)$.

In [ ]:
labels   = list(profiles_map.keys())
K        = len(labels)
node_ids = list(msi.nodelist)
N        = len(node_ids)

P_mat = np.stack([p[profiles_map[l]] for l in labels], axis=1)   # (N, K)

eps = np.finfo(float).tiny
W         = P_mat / np.maximum(P_mat.sum(axis=1, keepdims=True), eps)
W_c       = (W - 1.0 / K).astype(np.float32)                     # centred barycentric coords
M         = P_mat.mean(axis=1)
relevance = (M * np.log(N * np.maximum(M, eps))).astype(np.float32)
dominant  = np.argmax(W, axis=1).astype(np.int8)

print(f"Embedding shape: {W_c.shape}  (N={N:,}, K={K})")
print(f"Relevance: min={relevance.min():.4f}  median={np.median(relevance):.4f}  max={relevance.max():.4f}")
print()
for k, label in enumerate(labels):
    n_dom = (dominant == k).sum()
    print(f"  {label:<25s}  dominant in {n_dom:>7,} / {N:,} nodes  ({100*n_dom/N:.1f}%)")

## §5  Inspect specificity (optional)

Specificity $= \log(P_k(v) / M(v))$ — PMI of node $v$ with label $k$.  
Shows the top-N most specific nodes per disease.

In [ ]:
TOP_N = 10

spec = np.log(np.maximum(W, eps) / np.maximum(M[:, np.newaxis], eps))  # (N, K)
spec_df = pd.DataFrame(spec, index=node_ids, columns=labels)
rel_s   = pd.Series(relevance, index=node_ids, name="relevance")

for label in labels:
    top = spec_df[label].nlargest(TOP_N)
    print(f"\n── Top {TOP_N} nodes by specificity to '{label}' ──")
    for nid, val in top.items():
        name = msi.node2name.get(nid, nid)
        ntype = msi.node2type.get(nid, "?")
        print(f"  {val:+.3f}  {name:<40s}  [{ntype}]  {nid}")

## §6  Save to disk

In [ ]:
out_dir   = "results/simplex"
os.makedirs(out_dir, exist_ok=True)
stem      = os.path.join(out_dir, NAME)
npz_path  = stem + ".npz"
json_path = stem + ".json"

np.savez_compressed(npz_path, W_c=W_c, relevance=relevance, dominant=dominant)

meta = {
    "name":       NAME,
    "labels":     labels,
    "profiles":   profiles_map,
    "node_ids":   node_ids,
    "node_names": [msi.node2name.get(n, str(n)) for n in node_ids],
    "node_types": [msi.node2type.get(n, "?")    for n in node_ids],
    "K":          K,
    "N":          N,
}
with open(json_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved {npz_path}   ({os.path.getsize(npz_path)/1e6:.1f} MB)")
print(f"Saved {json_path}  ({os.path.getsize(json_path)/1e6:.1f} MB)")
print(f"\nRun the viewer:")
print(f"  python interactibles/simplex_3d.py --data {stem}")